# Section 2

## This is to ensure you have the proper tools installed

In [ ]:
%pip install pandas
%pip install duckdb
%pip install mysql-connector-python
%pip install kagglehub

## This checks if the necessary files are accessible and available

If the csv file can not be found download here and make sure it is in the correct directory:

https://drive.google.com/file/d/1B0P20CU9JP1I-jj0sgnq4QRvrtRLgUxF/view?usp=drive_link

In [ ]:
from pathlib import Path 
import os

csv_file = "IMDB Movie Big Dataset.csv"
current_dir = Path(os.getcwd())

if (current_dir / csv_file).is_file():
    csv_file = (current_dir / csv_file)
    print("CSV present")
else:
    import kagglehub
    cached_path = Path(kagglehub.dataset_download("shubhamchandra235/imdb-and-tmdb-movie-metadata-big-dataset-1m"))
    if cached_path.is_file():
        csv_file = cached_path
    elif cached_path.is_dir():
        csv_file = list(cached_path.glob("*.csv"))[0]
        target_dir = str(current_dir) + "/IMDB Movie Big Dataset.csv"
        os.rename(csv_file, target_dir)
        csv_file = target_dir
        print("Successfully downloaded and moved file")

In [ ]:
import ast
import sqlite3
import pandas as pd

def col_to_list(col):
    if pd.isna(col) or col == '':
        return []
    col = str(col).strip()
    try:
        col_list = ast.literal_eval(col)
        if isinstance(col_list, list):
            return col_list
        else:
            return [str(col_list).strip()]
    except (ValueError, SyntaxError):
        return [i.strip() for i in col.split(',') if i.strip()]

# Turns columnns from string representation to array 

#Small version for testing queries
#df = pd.read_csv(csv_file, nrows=100)

#Full db
df = pd.read_csv(csv_file)

# Renames variables to better names
df.rename(columns={
    'AverageRating':'average_rating',
    'genres_list':'genres',
}, inplace=True)
df.columns = df.columns.str.lower()

column_list = [
    "production_companies",
    "production_countries",
    "spoken_languages",
    "keywords",
    "producers",
    "genres",
    "cast_list",
    "all_combined_keywords"
]

for l in column_list:
    df[l] = df[l].apply(col_to_list)

conn = sqlite3.connect("IMDB Dataset.sqlite")
df_main = df.drop(columns=column_list)
df_main.to_sql("movies", conn, if_exists="replace", index=False)

In [ ]:
def create_junction_tables(df, col_id, col_list):
    junctions = {}
    for col in col_list:
        junction_df = (
            df[[col_id, col]]
                .explode(col)
                .dropna(subset=[col])
                .rename(columns={col_id: "movie_id", col:col})
        )
        junctions[col] = junction_df
    return junctions

junctions = create_junction_tables(df, col_id="id", col_list=column_list)

for col, junction_df in junctions.items():
    table_name = f"movie_{col}"
    junction_df.to_sql(table_name, conn, if_exists="replace", index=False)

In [ ]:
# Connect and show format of dataset
conn = sqlite3.connect("IMDB Dataset.sqlite")
movies_df = pd.read_sql("SELECT * FROM movies", conn)
movies_df.head(10)

## Implementations

In [ ]:
# Imports for convenience
from pathlib import Path 
import os
import ast
import sqlite3
import pandas as pd
import findspark
import csv
import io
import time

conn = sqlite3.connect("IMDB Dataset.sqlite")
numRuns = 10

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/spark-3.5.1-bin-hadoop3"


findspark.init("spark-3.5.1-bin-hadoop3/")

from pyspark import SparkContext
import multiprocessing

In [ ]:
# Step 1

timeAverage = 0


for i in range(numRuns):
    start = time.time()
    
    query = """
    SELECT title, revenue, vote_average, star1 AS Value FROM movies
    UNION ALL
    SELECT title, revenue, vote_average, star2 AS Value FROM movies
    UNION ALL
    SELECT title, revenue, vote_average, star3 AS Value FROM movies
    UNION ALL
    SELECT title, revenue, vote_average, star4 AS Value FROM movies;
    """
    
    melted_df = pd.read_sql(query, conn)
    melted_df.head(10)
    
    timeAverage += time.time() - start

timeAverage /= numRuns
print(timeAverage)

In [ ]:
#Step 2
averageTime = 0

for i in range(numRuns):

    start = time.time()
    
    query = """
    SELECT name, AVG(score) AS weighted_actor_score
    FROM (
       SELECT star1 AS name, vote_average AS score FROM movies
       UNION ALL
       SELECT star2 AS name, vote_average AS score FROM movies
       UNION ALL
       SELECT star3 AS name, vote_average AS score FROM movies
       UNION ALL
       SELECT star4 AS name, vote_average AS score FROM movies
    ) AS combined
    GROUP BY name;"""
    
    weighted_actor_df = pd.read_sql(query, conn)

    timeAverage += time.time() - start
timeAverage /= numRuns
print(timeAverage)

In [ ]:
weighted_actor_df.head(10)

In [ ]:
# Step 3

timeAverage = 0


for i in range(numRuns):

    start = time.time()
    
    film_rating_calc = melted_df.merge(weighted_actor_df, left_on='Value', right_on='name', how='left')
    film_rating_calc['actor_contribution'] = film_rating_calc['revenue'] / (film_rating_calc['weighted_actor_score'] + 1)
    
    film_rating_calc = (film_rating_calc.groupby('title')['actor_contribution']
                        .mean()
                        .reset_index()
                        .rename(columns={'actor_contribution': 'film_rating_calculation'}))
    
    timeAverage += time.time() - start
timeAverage /= numRuns
print(timeAverage)

In [ ]:
# Step 4
timeAverage = 0

for i in range(numRuns):
    start = time.time()
    query = """
    SELECT id, title, genres as genre
    FROM movies  
    INNER JOIN movie_genres ON (movies.id = movie_genres.movie_id)"""
    movie_genres_df = pd.read_sql(query, conn)
    
    step4 = melted_df.merge(movie_genres_df[['title', 'genre']], on='title', how='inner')
    step4 = step4.merge(weighted_actor_df, left_on='Value', right_on='name', how='left')
    timeAverage += time.time() - start

timeAverage /= numRuns
print(timeAverage)

In [ ]:
# Step 5
timeAverage = 0

for i in range(numRuns):
    start = time.time()
    step5 = step4.merge(film_rating_calc, on='title', how='left')
    step5 = step5.drop(columns='name')
    step5 = (step5.groupby('genre')['film_rating_calculation']
             .mean()
             .reset_index()
             .rename(columns={'film_rating_calculation': 'genre_score'}))
    timeAverage += time.time() - start
timeAverage /= numRuns
print(timeAverage)

In [ ]:
step5.head(25)

Spark Implementations of Step 1 and 4

In [ ]:
#Spark Implementation of Step 1 along with demonstration

start = time.time()

#sc = SparkContext("local[*]")
rdd = sc.textFile("IMDB Movie Big Dataset.csv")

#Remove header line, split the CSV into rows (Reduce)
header = rdd.first()
data = rdd.filter(lambda row: row != header)
rows = data.map(lambda line: next(csv.reader(io.StringIO(line))))

melted_rdd = rows.flatMap(lambda row: [
    (row[0], row[6], row[2], row[30]), 
    (row[0], row[6], row[2], row[31]),
    (row[0], row[6], row[2], row[32]),
    (row[0], row[6], row[2], row[33])
])

print(melted_rdd.take(4))
print(time.time() - start)


In [ ]:
#Setup step to turn previous steps into an RDD to begin spark. In real situation would already be RDD due to using spark for everything not just 2 steps
weighted_actor_df.to_csv("actor.csv", index=False, header=False)
weighted_actor_rdd = sc.textFile("actor.csv").map(lambda line: line.split(","))
weighted_actor_rdd.cache()
weighted_actor_rdd.count()

#Step 4
# First maps each full row to an ID and list of genres. Then maps each list into its elements so we have ID genre pairs
movie_genres_rdd = rows.flatMap(lambda row: [
    (row[0], row[38]), 
]).mapValues(lambda x: x[1:len(x)-1]).flatMapValues(lambda x: x.split(","))
joined_rdd = melted_rdd.map(lambda x: (x[0], x[3])).join(movie_genres_rdd)
joined_rdd = joined_rdd.map(lambda x: (x[1][0], [x[0], x[1][1]])).join(weighted_actor_rdd)
# Actually performing the spark operations
start = time.time()
print(joined_rdd.take(5))
print(time.time() - start)